In [2]:
import sys
sys.path.append('../')
from preload_data import load_empathetic_data

train, val, test, vocab = load_empathetic_data('../data')

加载数据: ../data/dataset_preproc.p
训练集: 40250 样本
验证集: 5734 样本
测试集: 5255 样本


In [10]:
import re

# Rename variable to avoid confusion with the dataset key 'target'
search_query = r"i saw this guy dump a pile of garbage from"
print(f"Searching for: '{search_query}'")
data_dict = {
    'trian': train, 'val': val, 'test': test
}
find_flag = False

# Iterate over datasets
for loader_name, loader in zip(['train', 'val', 'test'], [train, val, test]):
    length = len(loader['context'])
    keylist = ['context', 'emotion', 'target', 'situation']
    
    print(f"Scanning {loader_name} set ({length} samples)...")
    
    for i in range(length):
        # RESET FLAG FOR EACH SAMPLE
        
        for key in keylist:
            reference = loader[key][i]
            combined = ''
            
            # LIST of strings (target, situation)
            if key in ['target', 'situation']:
                combined = ' '.join(reference)
                if re.search(search_query, combined, re.IGNORECASE):
                    find_flag = True
            
            # EMOTION (String/Numpy String)
            elif key == 'emotion':
                combined = str(reference)
                if re.search(search_query, combined, re.IGNORECASE):
                    find_flag = True

            # CONTEXT (List of List of strings)
            elif key == 'context':
                for inner_ctx in reference:
                    combined = ' '.join(inner_ctx)
                    if re.search(search_query, combined, re.IGNORECASE):
                        find_flag = True
                        break # Stop checking context turns if found
            
            if find_flag: break # Stop checking keys if found in this sample

        if find_flag:
            print(f'Match found in {loader_name} set, idx {i}')
            # sys.exit(0)
            break
    if find_flag: break


Searching for: 'i saw this guy dump a pile of garbage from'
Scanning train set (40250 samples)...
Match found in train set, idx 28901


In [11]:
# Retrieve the correct loader based on the found name
# Re-define mapping to ensure 'train' key is correct (fixing previous typo 'trian')
lname, pos = loader_name, i
loaders_map = {'train': train, 'val': val, 'test': test}
current_loader = loaders_map.get(lname)

if current_loader:
    print(f"🔎 INSPECTING: [{lname}] set, Index [{pos}]")
    print("=" * 80)
    
    # 1. Emotion (Scalar)
    emotion = current_loader['emotion'][pos]
    print(f"❤️  EMOTION: {emotion}\n")
    
    # 2. Situation (List of tokens)
    situation_tokens = current_loader['situation'][pos]
    situation_str = " ".join(situation_tokens)
    print(f"📝 SITUATION:\n    {situation_str}\n")
    
    # 3. Context (List of List of tokens)
    print(f"💬 CONTEXT (Dialogue History):")
    context_turns = current_loader['context'][pos]
    for idx, turn_tokens in enumerate(context_turns):
        turn_str = " ".join(turn_tokens)
        # Assuming alternating speakers, though dataset specific
        print(f"    [Turn {idx+1}]: {turn_str}")
        
    # 4. Target (List of tokens)
    target_tokens = current_loader['target'][pos]
    target_str = " ".join(target_tokens)
    print(f"\n🎯 TARGET RESPONSE:\n    {target_str}")
    
    print("=" * 80)
else:
    print(f"Error: Loader '{lname}' not found.")

🔎 INSPECTING: [train] set, Index [28901]
❤️  EMOTION: disgusted

📝 SITUATION:
    i saw this guy dump a pile of garbage from his truck on the freeway . it was so messed up and dangerous for other drivers .

💬 CONTEXT (Dialogue History):
    [Turn 1]: i just recently witnessed this guy dump a pile of garbage from his truck on the freeway .

🎯 TARGET RESPONSE:
    garbage collection most important


In [8]:
ctx = train['context'][30606]
for idx, turn_tokens in enumerate(ctx):
        turn_str = " ".join(turn_tokens)
        # Assuming alternating speakers, though dataset specific
        print(f"    [Turn {idx+1}]: {turn_str}")

    [Turn 1]: i have been having a rough relationship lately , so tomorrow i plan to break up with my girlfriend .
    [Turn 2]: oh no ... i hope it is for the best
    [Turn 3]: well , i know how i feel and why it is not working . so i 'll simply tell it like it is , it should n't be too bad .


In [3]:
train.keys()

dict_keys(['context', 'target', 'emotion', 'situation', 'emotion_context', 'utt_cs', 'comet_cxt', 'comet_sit', 'comet_trg'])

In [ ]:
# Print samples in simple nested text format
def print_sample(data, index):
    print(f"{'='*60}")
    print(f"Sample #{index}")
    print(f"{'='*60}")
    
    # Emotion
    print(f"Emotion: {data['emotion'][index]}")
    
    # Situation
    situation = " ".join(data['situation'][index])
    print(f"\nSituation:\n  {situation}")
    
    # Context (dialogue history)
    print(f"\nContext (Dialogue History):")
    for i, turn in enumerate(data['context'][index]):
        turn_str = " ".join(turn)
        role = "User" if i % 2 == 0 else "Assistant"
        print(f"  Turn {i+1} [{role}]: {turn_str}")
    
    # Target
    target = " ".join(data['target'][index])
    print(f"\nTarget Response:\n  {target}")
    print()

# Print first few samples
for i in range(3):
    print_sample(train, i)

In [1]:
import sys
sys.path.append('../')
from src.utils.data.loader import prepare_data_seq
from transformers import AutoTokenizer
import os
from ZGeneration.config_gen import GenTrainingConfig

download_str: str = os.path.expanduser(r'~/Documents/LLModel')
model_name = 'llama3.1-8B-Instruct'
dl_path = os.path.join(download_str, model_name)

tokenizer = AutoTokenizer.from_pretrained(dl_path, trust_remote_code = False)

import re

template = tokenizer.chat_template
# Remove the date preamble section
template = re.sub(
    r'\{\{-\s*"Cutting Knowledge Date:.*?\}\}\s*\{\{-\s*"Today Date:.*?\}\}',
    '',
    template,
    flags=re.DOTALL
)
tokenizer.chat_template = template
if tokenizer.pad_token is None:
    # tokenizer.pad_token = tokenizer.eos_token
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})

config = GenTrainingConfig()
config.batch_size = 4

config.__post_init__()

train_loader, val_loader, test_loader, vocab, emo_dict, dataset_train = prepare_data_seq(tokenizer, config)

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOADING empathetic_dialogue
[situation]: i remember going to the fireworks with my best friend . there was a lot of people , but it only felt like us in the world .
[emotion]: sentimental
[context]: ['i remember going to see the fireworks with my best friend . it was the first time we ever spent time alone together . although there was a lot of people , we felt like the only people in the world .']
[target]: was this a friend you were in love with , or just a best friend ?
 
[situation]: i remember going to the fireworks with my best friend . there was a lot of people , but it only felt like us in the world .
[emotion]: sentimental
[context]: ['i remember going to see the fireworks with my best friend . it was the first time we ever spent time alone together . although there was a lot of people , we felt like the only people in the world .', 'was this a friend you were in love with , or just a best friend ?', 'this was a best friend . i miss her .']
[target]: where has she gone ?
 
[si

In [2]:
test1 = next(iter(dataset_train))
ptest1 = next(iter(train_loader))

In [5]:
for i in range(4):
    ctx, target, prompt = ptest1['input_batch'][i], ptest1['target_batch'][i], ptest1['prompt_ids'][i]
    ctx_sentences, target_sentences, prompt_stns = tokenizer.batch_decode(ctx, skip_special_tokens=True), tokenizer.batch_decode(target, skip_special_tokens=True), tokenizer.batch_decode(ptest1['prompt_ids'][i], skip_special_tokens=True)
    print(''.join(ctx_sentences))
    print("[Target]", ''.join(target_sentences))
    print("[Ctx]", ''.join(prompt_stns), '\n\n')

system

You are the assistant trying to show your empathy to the user during the conversation. Please don't over reply to the user's message (i.e., no need to use so many sentences.). Reply to the user's message as naturally as possible.user

my boyfriend and i celebrated our sixth anniversary the other week. i was pretty sure he was planning to propose because of the nice plans he had made- and he was. for a couple of days leading up to it i was so excited and just could hardly wait.assistant

wow that is exciting for sure. you must have been filled with anticipation.user

i was, even though i was a bit just antsy to find out if i was even right. i was really happy!assistant

i certainly hope he did propose and that you are still happily living.
[Target] i certainly hope he did propose and that you are still happily living.
[Ctx] system

You are the assistant trying to show your empathy to the user during the conversation. Please don't over reply to the user's message (i.e., no need t

In [5]:
sample_position = ptest1['labels'][1]
tokenizer.decode(sample_position[sample_position!=-100])

'my husband has that issue. are you shorter than average?<|eot_id|>'

In [4]:
ptest1['program_label']

[14, 4, 24, 31]